# 01. Data Integration and Dataset Understanding

In this notebook, we will understand and organize the MoA Prediction dataset before doing EDA, feature engineering, or model training.

The main goals of this notebook are:

- load all raw dataset files,
- check the shape of each file,
- understand the role of each table,
- identify important columns,
- check duplicate `sig_id` values,
- check missing values,
- check whether IDs match across files,
- create a safe data integration plan.

At this stage, we will not train any model.  
We will only prepare the dataset correctly.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [2]:
# Detect project root safely
PROJECT_ROOT = Path.cwd()

# If notebook is running from the notebooks folder, go one level up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DATA_DIR)
print("Interim data folder:", INTERIM_DATA_DIR)

Project root: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response
Raw data folder: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\raw
Interim data folder: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\interim


## 1. Project Setup and Raw Data Loading

This section sets up the notebook, checks whether the required raw files are available, loads the CSV files, and creates the first shape overview.

### 1.1 Check Required Raw Files

Before loading the dataset, we first check whether all required CSV files are available inside the `data/raw/` folder.

This prevents errors later and confirms that the project folder is correctly organized.

In [3]:
required_files = [
    "train_features.csv",
    "test_features.csv",
    "train_targets_scored.csv",
    "train_targets_nonscored.csv",
    "train_drug.csv",
    "sample_submission.csv",
]

file_check = []

for file_name in required_files:
    file_path = RAW_DATA_DIR / file_name
    file_check.append({
        "file_name": file_name,
        "exists": file_path.exists(),
        "path": str(file_path)
    })

file_check_df = pd.DataFrame(file_check)
file_check_df

,file_name,exists,path
0,train_features.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
1,test_features.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
2,train_targets_scored.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
3,train_targets_nonscored.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
4,train_drug.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
5,sample_submission.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...


### 1.2 Load Raw Dataset Files

Now we load all raw CSV files into pandas DataFrames.

At this stage, we only load the data.  
We will not merge anything yet because we first need to check shapes, columns, duplicate IDs, and ID alignment.

In [4]:
train_features = pd.read_csv(RAW_DATA_DIR / "train_features.csv")
test_features = pd.read_csv(RAW_DATA_DIR / "test_features.csv")

train_targets_scored = pd.read_csv(RAW_DATA_DIR / "train_targets_scored.csv")
train_targets_nonscored = pd.read_csv(RAW_DATA_DIR / "train_targets_nonscored.csv")

train_drug = pd.read_csv(RAW_DATA_DIR / "train_drug.csv")
sample_submission = pd.read_csv(RAW_DATA_DIR / "sample_submission.csv")

print("All files loaded successfully.")

All files loaded successfully.


### 1.3 First Look at Dataset Shapes

After loading the files, we check the number of rows and columns in each table.

This helps us understand which files contain input features, which files contain targets, and whether the row counts match as expected.

In [5]:
file_overview = pd.DataFrame({
    "file_name": [
        "train_features",
        "test_features",
        "train_targets_scored",
        "train_targets_nonscored",
        "train_drug",
        "sample_submission",
    ],
    "rows": [
        train_features.shape[0],
        test_features.shape[0],
        train_targets_scored.shape[0],
        train_targets_nonscored.shape[0],
        train_drug.shape[0],
        sample_submission.shape[0],
    ],
    "columns": [
        train_features.shape[1],
        test_features.shape[1],
        train_targets_scored.shape[1],
        train_targets_nonscored.shape[1],
        train_drug.shape[1],
        sample_submission.shape[1],
    ],
})

file_overview

,file_name,rows,columns
0,train_features,23814,876
1,test_features,3982,876
2,train_targets_scored,23814,207
3,train_targets_nonscored,23814,403
4,train_drug,23814,2
5,sample_submission,3982,207


### 1.4 Dataset File Roles

The MoA dataset is divided into input feature files, target files, drug information, and submission format.

- `train_features.csv` is the main training input table. It contains metadata, gene expression features, and cell viability features.
- `test_features.csv` is the main test input table. It has the same feature columns as `train_features.csv`, but it does not contain target labels.
- `train_targets_scored.csv` contains the main target labels used for model training and Kaggle scoring.
- `train_targets_nonscored.csv` contains extra target labels. These are not required for baseline modeling, but they may be useful later for auxiliary learning.
- `train_drug.csv` contains anonymous drug IDs for training samples only.
- `sample_submission.csv` shows the required final submission format.

At this stage, we only understand the files. We will not merge anything yet.

## 2. Column and Feature Group Understanding

This section identifies the important column groups and checks whether the train and test feature columns are consistent.

### 2.1 Identify Column Groups

Now we separate the columns into meaningful groups.

The main column groups are:

- ID column: `sig_id`
- metadata columns: `cp_type`, `cp_time`, `cp_dose`
- gene expression columns: columns starting with `g-`
- cell viability columns: columns starting with `c-`
- scored target columns: target labels from `train_targets_scored.csv`
- nonscored target columns: auxiliary labels from `train_targets_nonscored.csv`

This step is important because later we will use different feature groups for EDA, feature engineering, and model training.

In [6]:
ID_COL = "sig_id"

metadata_features = ["cp_type", "cp_time", "cp_dose"]

gene_features = [col for col in train_features.columns if col.startswith("g-")]
cell_features = [col for col in train_features.columns if col.startswith("c-")]

scored_target_features = [
    col for col in train_targets_scored.columns 
    if col != ID_COL
]

nonscored_target_features = [
    col for col in train_targets_nonscored.columns 
    if col != ID_COL
]

feature_group_summary = pd.DataFrame({
    "feature_group": [
        "ID column",
        "metadata features",
        "gene expression features",
        "cell viability features",
        "scored target labels",
        "nonscored target labels",
    ],
    "count": [
        1,
        len(metadata_features),
        len(gene_features),
        len(cell_features),
        len(scored_target_features),
        len(nonscored_target_features),
    ]
})

feature_group_summary

,feature_group,count
0,ID column,1
1,metadata features,3
2,gene expression features,772
3,cell viability features,100
4,scored target labels,206
5,nonscored target labels,402


### 2.2 Check Important Column Names

Before doing validation or merging, we quickly check whether the expected key and metadata columns exist.

This helps prevent future errors during merging and feature engineering.

In [7]:
important_column_check = {
    "sig_id_in_train_features": ID_COL in train_features.columns,
    "sig_id_in_test_features": ID_COL in test_features.columns,
    "sig_id_in_train_targets_scored": ID_COL in train_targets_scored.columns,
    "sig_id_in_train_targets_nonscored": ID_COL in train_targets_nonscored.columns,
    "sig_id_in_train_drug": ID_COL in train_drug.columns,
    "sig_id_in_sample_submission": ID_COL in sample_submission.columns,
    "metadata_columns_exist": all(col in train_features.columns for col in metadata_features),
    "drug_id_exists": "drug_id" in train_drug.columns,
}

important_column_check

{'sig_id_in_train_features': True,
 'sig_id_in_test_features': True,
 'sig_id_in_train_targets_scored': True,
 'sig_id_in_train_targets_nonscored': True,
 'sig_id_in_train_drug': True,
 'sig_id_in_sample_submission': True,
 'metadata_columns_exist': True,
 'drug_id_exists': True}

### 2.3 Check Train and Test Feature Consistency

The training and test feature tables should contain the same input feature columns.

This is important because the model will be trained on `train_features.csv` and later used to predict on `test_features.csv`.

If train and test feature columns do not match, model training or prediction will fail.

In [8]:
train_feature_cols = set(train_features.columns) - {ID_COL}
test_feature_cols = set(test_features.columns) - {ID_COL}

missing_in_test = train_feature_cols - test_feature_cols
extra_in_test = test_feature_cols - train_feature_cols

column_consistency_report = {
    "train_feature_count_without_id": len(train_feature_cols),
    "test_feature_count_without_id": len(test_feature_cols),
    "missing_in_test": list(missing_in_test),
    "extra_in_test": list(extra_in_test),
    "columns_match": len(missing_in_test) == 0 and len(extra_in_test) == 0,
}

column_consistency_report

{'train_feature_count_without_id': 875,
 'test_feature_count_without_id': 875,
 'missing_in_test': [],
 'extra_in_test': [],
 'columns_match': True}

## 3. Data Quality and ID Validation

This section checks duplicate IDs, missing values, ID matching, row alignment, missing or extra IDs, and target validity before any safe merge.

### 3.1 Check Duplicate `sig_id` Values

The `sig_id` column is the unique identifier for each sample.

Before merging any files, we must confirm that each `sig_id` appears only once in every table. If duplicate IDs exist, merging can create duplicated rows and damage the dataset structure.

This check helps us make sure that the data can be safely connected using `sig_id`.

In [9]:
# Store all loaded DataFrames in one dictionary for repeated checks

files = {
    "train_features": train_features,
    "test_features": test_features,
    "train_targets_scored": train_targets_scored,
    "train_targets_nonscored": train_targets_nonscored,
    "train_drug": train_drug,
    "sample_submission": sample_submission,
}

files.keys()

dict_keys(['train_features', 'test_features', 'train_targets_scored', 'train_targets_nonscored', 'train_drug', 'sample_submission'])

In [10]:
duplicate_id_report = []

for name, df in files.items():
    duplicate_id_report.append({
        "file_name": name,
        "total_rows": df.shape[0],
        "unique_sig_id": df[ID_COL].nunique(),
        "duplicate_sig_id_count": df[ID_COL].duplicated().sum(),
        "is_sig_id_unique": df[ID_COL].is_unique,
    })

duplicate_id_report = pd.DataFrame(duplicate_id_report)
duplicate_id_report

,file_name,total_rows,unique_sig_id,duplicate_sig_id_count,is_sig_id_unique
0,train_features,23814,23814,0,True
1,test_features,3982,3982,0,True
2,train_targets_scored,23814,23814,0,True
3,train_targets_nonscored,23814,23814,0,True
4,train_drug,23814,23814,0,True
5,sample_submission,3982,3982,0,True


### 3.2 Check Missing Values

Now we check whether any file contains missing values.

Missing values can cause problems during feature engineering and model training. Even if we expect this dataset to be clean, we should always check it before moving forward.

In [11]:
missing_value_report = []

for name, df in files.items():
    total_missing = df.isna().sum().sum()
    columns_with_missing = (df.isna().sum() > 0).sum()
    
    missing_value_report.append({
        "file_name": name,
        "total_missing_values": total_missing,
        "columns_with_missing": columns_with_missing,
        "missing_percentage": round((total_missing / df.size) * 100, 4),
    })

missing_value_report = pd.DataFrame(missing_value_report)
missing_value_report

,file_name,total_missing_values,columns_with_missing,missing_percentage
0,train_features,0,0,0.0
1,test_features,0,0,0.0
2,train_targets_scored,0,0,0.0
3,train_targets_nonscored,0,0,0.0
4,train_drug,0,0,0.0
5,sample_submission,0,0,0.0


In [12]:
for name, df in files.items():
    missing_by_column = df.isna().sum()
    missing_by_column = missing_by_column[missing_by_column > 0].sort_values(ascending=False)
    
    if len(missing_by_column) > 0:
        print(f"\n{name} columns with missing values:")
        display(missing_by_column)

### 3.3 Check ID Matching Across Files

Before merging tables, we need to confirm that the same `sig_id` values exist across related files.

For the training data, `train_features`, `train_targets_scored`, `train_targets_nonscored`, and `train_drug` should contain the same training sample IDs.

For the test data, `test_features` and `sample_submission` should contain the same test sample IDs.

This check confirms whether the files can be safely connected using `sig_id`.

In [13]:
id_matching_report = pd.DataFrame({
    "check": [
        "train_features vs train_targets_scored",
        "train_features vs train_targets_nonscored",
        "train_features vs train_drug",
        "test_features vs sample_submission",
    ],
    "same_id_set": [
        set(train_features[ID_COL]) == set(train_targets_scored[ID_COL]),
        set(train_features[ID_COL]) == set(train_targets_nonscored[ID_COL]),
        set(train_features[ID_COL]) == set(train_drug[ID_COL]),
        set(test_features[ID_COL]) == set(sample_submission[ID_COL]),
    ],
})

id_matching_report

,check,same_id_set
0,train_features vs train_targets_scored,True
1,train_features vs train_targets_nonscored,True
2,train_features vs train_drug,True
3,test_features vs sample_submission,True


### 3.4 Check Row Alignment

After checking that the same IDs exist across files, we also check whether they are in the same row order.

This is important because some people directly join or compare tables by row position. However, row order should never be trusted blindly.

Even if row order is different, we can still safely merge using `sig_id`.

In [14]:
row_alignment_report = pd.DataFrame({
    "check": [
        "train_features vs train_targets_scored",
        "train_features vs train_targets_nonscored",
        "train_features vs train_drug",
        "test_features vs sample_submission",
    ],
    "same_row_order": [
        (train_features[ID_COL].values == train_targets_scored[ID_COL].values).all(),
        (train_features[ID_COL].values == train_targets_nonscored[ID_COL].values).all(),
        (train_features[ID_COL].values == train_drug[ID_COL].values).all(),
        (test_features[ID_COL].values == sample_submission[ID_COL].values).all(),
    ],
})

row_alignment_report

,check,same_row_order
0,train_features vs train_targets_scored,True
1,train_features vs train_targets_nonscored,True
2,train_features vs train_drug,True
3,test_features vs sample_submission,True


### 3.5 Check Missing or Extra IDs

If any ID matching check fails, we need to know which IDs are missing or extra.

This step creates a small diagnostic function. If everything is already matched, all missing and extra counts should be zero.

In [15]:
def compare_id_sets_clean(left_df, right_df, left_name, right_name, id_col=ID_COL):
    left_ids = set(left_df[id_col])
    right_ids = set(right_df[id_col])
    
    return {
        "comparison": f"{left_name} vs {right_name}",
        "left_table": left_name,
        "right_table": right_name,
        "left_ids_not_in_right": len(left_ids - right_ids),
        "right_ids_not_in_left": len(right_ids - left_ids),
        "ids_match": left_ids == right_ids,
    }


missing_extra_id_report = pd.DataFrame([
    compare_id_sets_clean(train_features, train_targets_scored, "train_features", "train_targets_scored"),
    compare_id_sets_clean(train_features, train_targets_nonscored, "train_features", "train_targets_nonscored"),
    compare_id_sets_clean(train_features, train_drug, "train_features", "train_drug"),
    compare_id_sets_clean(test_features, sample_submission, "test_features", "sample_submission"),
])

missing_extra_id_report

,comparison,left_table,right_table,left_ids_not_in_right,right_ids_not_in_left,ids_match
0,train_features vs train_targets_scored,train_features,train_targets_scored,0,0,True
1,train_features vs train_targets_nonscored,train_features,train_targets_nonscored,0,0,True
2,train_features vs train_drug,train_features,train_drug,0,0,True
3,test_features vs sample_submission,test_features,sample_submission,0,0,True


### 3.6 Check Target Validity

The MoA target columns should contain only binary values.

For both scored and nonscored targets:

- `0` means the target is not active.
- `1` means the target is active.

Before modeling, we need to confirm that there are no unexpected values in the target files.

In [16]:
scored_unique_values = np.unique(train_targets_scored[scored_target_features].values)
nonscored_unique_values = np.unique(train_targets_nonscored[nonscored_target_features].values)

target_validity_report = {
    "scored_unique_values": scored_unique_values.tolist(),
    "nonscored_unique_values": nonscored_unique_values.tolist(),
    "scored_targets_are_binary": set(scored_unique_values).issubset({0, 1}),
    "nonscored_targets_are_binary": set(nonscored_unique_values).issubset({0, 1}),
}

target_validity_report

{'scored_unique_values': [0, 1],
 'nonscored_unique_values': [0, 1],
 'scored_targets_are_binary': True,
 'nonscored_targets_are_binary': True}

## 4. Metadata, Target, and Control Validation

This section validates the metadata values, target-submission column consistency, and the behavior of control samples.

### 4.1 Check Metadata Column Values

The metadata columns describe the treatment setup for each sample.

We check the unique values of:

- `cp_type`: treatment type
- `cp_time`: treatment duration
- `cp_dose`: treatment dose

This helps us confirm that the metadata columns contain expected categories before EDA and feature engineering.

In [17]:
metadata_value_report = {
    "cp_type_unique_values": sorted(train_features["cp_type"].unique().tolist()),
    "cp_time_unique_values": sorted(train_features["cp_time"].unique().tolist()),
    "cp_dose_unique_values": sorted(train_features["cp_dose"].unique().tolist()),
}

metadata_value_report

{'cp_type_unique_values': ['ctl_vehicle', 'trt_cp'],
 'cp_time_unique_values': [24, 48, 72],
 'cp_dose_unique_values': ['D1', 'D2']}

### 4.2 Check Metadata Distribution

Now we check how many samples exist in each metadata category.

This is not full EDA yet.  
It is only a basic dataset understanding step to see whether the treatment types, time points, and dose levels are distributed properly.

In [18]:
metadata_distribution = {
    "cp_type_distribution": train_features["cp_type"].value_counts().to_dict(),
    "cp_time_distribution": train_features["cp_time"].value_counts().sort_index().to_dict(),
    "cp_dose_distribution": train_features["cp_dose"].value_counts().to_dict(),
}

metadata_distribution

{'cp_type_distribution': {'trt_cp': 21948, 'ctl_vehicle': 1866},
 'cp_time_distribution': {24: 7772, 48: 8250, 72: 7792},
 'cp_dose_distribution': {'D1': 12147, 'D2': 11667}}

### 4.3 Check Train and Test Metadata Consistency

The model will be trained on `train_features` and later used on `test_features`.

So the metadata categories in train and test should be consistent.

Here we check whether `cp_type`, `cp_time`, and `cp_dose` contain the same types of values in both train and test data.

In [19]:
metadata_consistency_report = []

for col in metadata_features:
    train_values = sorted(train_features[col].unique().tolist())
    test_values = sorted(test_features[col].unique().tolist())
    
    metadata_consistency_report.append({
        "column": col,
        "train_unique_values": train_values,
        "test_unique_values": test_values,
        "same_values": set(train_values) == set(test_values),
        "values_in_train_not_test": list(set(train_values) - set(test_values)),
        "values_in_test_not_train": list(set(test_values) - set(train_values)),
    })

metadata_consistency_report = pd.DataFrame(metadata_consistency_report)
metadata_consistency_report

,column,train_unique_values,test_unique_values,same_values,values_in_train_not_test,values_in_test_not_train
0,cp_type,"[ctl_vehicle, trt_cp]","[ctl_vehicle, trt_cp]",True,[],[]
1,cp_time,"[24, 48, 72]","[24, 48, 72]",True,[],[]
2,cp_dose,"[D1, D2]","[D1, D2]",True,[],[]


### 4.4 Check Target and Submission Column Consistency

The final submission must predict the same target columns that exist in `train_targets_scored.csv`.

Here we check whether the scored target columns match the columns in `sample_submission.csv`.

This is important because the final prediction file must have exactly the same target columns and column order.

In [20]:
sample_submission_target_features = [
    col for col in sample_submission.columns 
    if col != ID_COL
]

target_submission_report = {
    "scored_target_count": len(scored_target_features),
    "submission_target_count": len(sample_submission_target_features),
    "missing_in_submission": list(set(scored_target_features) - set(sample_submission_target_features)),
    "extra_in_submission": list(set(sample_submission_target_features) - set(scored_target_features)),
    "same_target_set": set(scored_target_features) == set(sample_submission_target_features),
    "same_target_order": scored_target_features == sample_submission_target_features,
}

target_submission_report

{'scored_target_count': 206,
 'submission_target_count': 206,
 'missing_in_submission': [],
 'extra_in_submission': [],
 'same_target_set': True,
 'same_target_order': True}

### 4.5 Check Control Sample Target Behavior

Control samples have `cp_type == "ctl_vehicle"`.

These samples should not have active MoA targets because they are vehicle control samples, not treated compound samples.

This check is important because later, during final prediction, we can set all target probabilities to zero for test rows where `cp_type == "ctl_vehicle"`.

In [21]:
control_ids = train_features.loc[
    train_features["cp_type"] == "ctl_vehicle",
    ID_COL
]

control_targets = train_targets_scored.loc[
    train_targets_scored[ID_COL].isin(control_ids),
    scored_target_features
]

control_report = {
    "control_sample_count": len(control_ids),
    "control_target_rows": control_targets.shape[0],
    "control_positive_target_count": int(control_targets.sum().sum()),
    "all_control_targets_zero": int(control_targets.sum().sum()) == 0,
}

control_report

{'control_sample_count': 1866,
 'control_target_rows': 1866,
 'control_positive_target_count': 0,
 'all_control_targets_zero': True}

## 5. Drug Information Review

This section reviews `train_drug.csv` to understand repeated drugs and prepare for possible drug-aware validation later.

### 5.1 Check Drug Information

The `train_drug.csv` file contains anonymous drug IDs for the training samples.

This file is useful for understanding repeated drugs and for possible drug-aware validation later.

However, `drug_id` should not be used directly in the baseline model because it is only available for the training data.

In [22]:
drug_report = {
    "train_drug_rows": train_drug.shape[0],
    "train_drug_columns": train_drug.shape[1],
    "unique_sig_id": train_drug[ID_COL].nunique(),
    "unique_drug_id": train_drug["drug_id"].nunique(),
    "duplicate_sig_id_count": train_drug[ID_COL].duplicated().sum(),
    "missing_drug_id_count": train_drug["drug_id"].isna().sum(),
}

drug_report

{'train_drug_rows': 23814,
 'train_drug_columns': 2,
 'unique_sig_id': 23814,
 'unique_drug_id': 3289,
 'duplicate_sig_id_count': np.int64(0),
 'missing_drug_id_count': np.int64(0)}

### 5.2 Check Repeated Drug Frequency

Some drugs appear multiple times in the training data.

This is important because repeated drugs can affect validation. If the same drug appears in both training and validation folds, the validation score may become slightly optimistic.

For the baseline model, we will not use `drug_id` as a feature. Later, we may use it for drug-aware validation.

In [23]:
drug_frequency_summary = train_drug["drug_id"].value_counts().describe()

drug_frequency_summary

count    3289.000000
mean        7.240499
std        35.901370
min         1.000000
25%         6.000000
50%         6.000000
75%         6.000000
max      1866.000000
Name: count, dtype: float64

In [24]:
top_repeated_drugs = (
    train_drug["drug_id"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_repeated_drugs.columns = ["drug_id", "sample_count"]

top_repeated_drugs

,drug_id,sample_count
0,cacb2b860,1866
1,87d714366,718
2,9f80f3f77,246
3,8b87a7a83,203
4,5628cb3ee,202
5,d08af5d4b,196
6,292ab2c28,194
7,d50f18348,186
8,d1b47f29d,178
9,67c879e79,19


## 6. Safe Data Integration

After completing the basic dataset checks, we can now safely connect the related tables.

We will use `sig_id` as the merge key because it uniquely identifies each sample.

In this step, we create three useful integrated tables:

- `train_main_scored`: training features + scored targets
- `train_with_drug`: training features + drug ID
- `train_aux_nonscored`: sample IDs + nonscored auxiliary targets

These merged tables are useful for checking, EDA, and future analysis.

For model training, we will still keep input features and target labels separate to avoid confusion.

In [26]:
train_main_scored = train_features.merge(
    train_targets_scored,
    on=ID_COL,
    how="left",
    validate="one_to_one"
)

train_with_drug = train_features.merge(
    train_drug,
    on=ID_COL,
    how="left",
    validate="one_to_one"
)

train_aux_nonscored = train_features[[ID_COL]].merge(
    train_targets_nonscored,
    on=ID_COL,
    how="left",
    validate="one_to_one"
)

print("Safe merged tables created successfully.")

Safe merged tables created successfully.


In [31]:
train_with_drug.head()

,sig_id,cp_type,cp_time,cp_dose,g-0,g-1,g-2,g-3,g-4,g-5,...,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99,drug_id
0,id_000644bb2,trt_cp,24,D1,1.0620,0.5577,-0.2479,-0.6208,-0.1944,-1.0120,...,0.2584,0.8076,0.5523,-0.1912,0.6584,-0.3981,0.2139,0.3801,0.4176,b68db1d53
1,id_000779bfc,trt_cp,72,D1,0.0743,0.4087,0.2991,0.0604,1.0190,0.5207,...,0.7543,0.4708,0.0230,0.2957,0.4899,0.1522,0.1241,0.6077,0.7371,df89a8e5a
2,id_000a6266a,trt_cp,48,D1,0.6280,0.5817,1.5540,-0.0764,-0.0323,1.2390,...,-0.6297,0.6103,0.0223,-1.3240,-0.3174,-0.6417,-0.2187,-1.4080,0.6931,18bb41b2c
3,id_0015fd391,trt_cp,48,D1,-0.5138,-0.2491,-0.2656,0.5288,4.0620,-0.8095,...,-0.6441,-5.6300,-1.3780,-0.8632,-1.2880,-1.6210,-0.8784,-0.3876,-0.8154,8c7f86626
4,id_001626bd3,trt_cp,72,D2,-0.3254,-0.4009,0.9700,0.6919,1.4180,-0.8244,...,0.0048,0.6670,1.0690,0.5523,-0.3031,0.1094,0.2885,-0.3786,0.7125,7cbed3131


### 6.1 Check Integrated Table Shapes

After merging, we check the shape of each integrated table.

This confirms that the number of rows did not change after merging.

If the row count changes, it may indicate duplicate IDs or incorrect merge behavior.

In [32]:
integrated_shape_report = pd.DataFrame({
    "table_name": [
        "train_main_scored",
        "train_with_drug",
        "train_aux_nonscored",
    ],
    "rows": [
        train_main_scored.shape[0],
        train_with_drug.shape[0],
        train_aux_nonscored.shape[0],
    ],
    "columns": [
        train_main_scored.shape[1],
        train_with_drug.shape[1],
        train_aux_nonscored.shape[1],
    ],
})

integrated_shape_report

,table_name,rows,columns
0,train_main_scored,23814,1082
1,train_with_drug,23814,877
2,train_aux_nonscored,23814,403


### 6.2 Check Missing Values After Merge

Even though the original files had no missing values, we check again after merging.

This confirms that all matching IDs were connected correctly and no target or drug information was lost during the merge.

In [34]:
post_merge_missing_report = pd.DataFrame({
    "table_name": [
        "train_main_scored",
        "train_with_drug",
        "train_aux_nonscored",
    ],
    "total_missing_values": [
        train_main_scored.isna().sum().sum(),
        train_with_drug.isna().sum().sum(),
        train_aux_nonscored.isna().sum().sum(),
    ],
    "columns_with_missing": [
        (train_main_scored.isna().sum() > 0).sum(),
        (train_with_drug.isna().sum() > 0).sum(),
        (train_aux_nonscored.isna().sum() > 0).sum(),
    ],
})

post_merge_missing_report

,table_name,total_missing_values,columns_with_missing
0,train_main_scored,0,0
1,train_with_drug,0,0
2,train_aux_nonscored,0,0


## 7. Baseline Modeling Structure

Now we prepare the clean baseline input and target structure.

For the baseline model:

- `X_train_base` will contain the training input features.
- `X_test_base` will contain the test input features.
- `y_scored_base` will contain only the 206 scored target labels.
- `y_nonscored_aux` will be saved separately for possible advanced auxiliary learning later.

We will not use `sig_id`, `drug_id`, or nonscored targets in the baseline model.

In [35]:
X_train_base = train_features.drop(columns=[ID_COL]).copy()
X_test_base = test_features.drop(columns=[ID_COL]).copy()

y_scored_base = train_targets_scored[scored_target_features].copy()
y_nonscored_aux = train_targets_nonscored[nonscored_target_features].copy()

baseline_structure_report = {
    "X_train_base_shape": X_train_base.shape,
    "X_test_base_shape": X_test_base.shape,
    "y_scored_base_shape": y_scored_base.shape,
    "y_nonscored_aux_shape": y_nonscored_aux.shape,
    "sig_id_removed_from_X_train": ID_COL not in X_train_base.columns,
    "sig_id_removed_from_X_test": ID_COL not in X_test_base.columns,
    "drug_id_used_in_baseline": "drug_id" in X_train_base.columns,
    "baseline_target_count": y_scored_base.shape[1],
    "auxiliary_target_count": y_nonscored_aux.shape[1],
}

baseline_structure_report


{'X_train_base_shape': (23814, 875),
 'X_test_base_shape': (3982, 875),
 'y_scored_base_shape': (23814, 206),
 'y_nonscored_aux_shape': (23814, 402),
 'sig_id_removed_from_X_train': True,
 'sig_id_removed_from_X_test': True,
 'drug_id_used_in_baseline': False,
 'baseline_target_count': 206,
 'auxiliary_target_count': 402}

### 7.1 Verify Baseline Feature Columns

Before saving the baseline structure, we quickly verify that the input feature columns are correct.

The baseline features should include:

- 3 metadata columns,
- 772 gene expression features,
- 100 cell viability features.

Total feature count should be 875.

In [36]:
baseline_feature_check = {
    "metadata_features_in_X": all(col in X_train_base.columns for col in metadata_features),
    "gene_feature_count_in_X": len([col for col in X_train_base.columns if col.startswith("g-")]),
    "cell_feature_count_in_X": len([col for col in X_train_base.columns if col.startswith("c-")]),
    "total_feature_count": X_train_base.shape[1],
    "same_train_test_columns": list(X_train_base.columns) == list(X_test_base.columns),
}

baseline_feature_check

{'metadata_features_in_X': True,
 'gene_feature_count_in_X': 772,
 'cell_feature_count_in_X': 100,
 'total_feature_count': 875,
 'same_train_test_columns': True}

## 8. Save Feature Groups and Validation Reports

In this section, we save important dataset information for future notebooks.

First, we save the feature groups:

- metadata features
- gene expression features
- cell viability features
- scored target labels
- nonscored target labels

This will help us reuse the same column groups in EDA, feature engineering, and modeling without redefining them manually.

In [37]:
feature_groups = {
    "id_col": ID_COL,
    "metadata_features": metadata_features,
    "gene_features": gene_features,
    "cell_features": cell_features,
    "scored_target_features": scored_target_features,
    "nonscored_target_features": nonscored_target_features,
}

feature_groups_path = INTERIM_DATA_DIR / "feature_groups.json"

with open(feature_groups_path, "w") as f:
    json.dump(feature_groups, f, indent=4)

print(f"Feature groups saved to: {feature_groups_path}")

Feature groups saved to: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\interim\feature_groups.json


### 8.1 Check Saved Feature Groups

After saving the feature groups, we load the JSON file again to confirm that it was saved correctly.

This is a simple safety check before moving to the next notebook.

In [38]:
with open(feature_groups_path, "r") as f:
    loaded_feature_groups = json.load(f)

feature_group_saved_check = {
    "id_col": loaded_feature_groups["id_col"],
    "metadata_feature_count": len(loaded_feature_groups["metadata_features"]),
    "gene_feature_count": len(loaded_feature_groups["gene_features"]),
    "cell_feature_count": len(loaded_feature_groups["cell_features"]),
    "scored_target_count": len(loaded_feature_groups["scored_target_features"]),
    "nonscored_target_count": len(loaded_feature_groups["nonscored_target_features"]),
}

feature_group_saved_check

{'id_col': 'sig_id',
 'metadata_feature_count': 3,
 'gene_feature_count': 772,
 'cell_feature_count': 100,
 'scored_target_count': 206,
 'nonscored_target_count': 402}

### 8.2 Save Data Validation Report

Now we save the main validation results from this notebook.

This report records the checks we completed, such as:

- file shapes,
- feature group counts,
- duplicate ID checks,
- missing value checks,
- ID matching,
- row alignment,
- target validity,
- metadata consistency,
- control sample behavior,
- drug information,
- merged table shapes,
- baseline structure.

This makes the project easier to review and reproduce later.

In [39]:
def make_json_serializable(obj):
    """
    Convert NumPy / pandas objects into JSON-friendly Python objects.
    """
    if isinstance(obj, dict):
        return {key: make_json_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(value) for value in obj]
    elif isinstance(obj, tuple):
        return [make_json_serializable(value) for value in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    elif isinstance(obj, pd.Series):
        return obj.to_dict()
    else:
        return obj


validation_report = {
    "file_overview": file_overview,
    "feature_group_summary": feature_group_summary,
    "column_consistency_report": column_consistency_report,
    "duplicate_id_report": duplicate_id_report,
    "missing_value_report": missing_value_report,
    "id_matching_report": id_matching_report,
    "row_alignment_report": row_alignment_report,
    "missing_extra_id_report": missing_extra_id_report,
    "target_validity_report": target_validity_report,
    "metadata_value_report": metadata_value_report,
    "metadata_distribution": metadata_distribution,
    "metadata_consistency_report": metadata_consistency_report,
    "target_submission_report": target_submission_report,
    "control_report": control_report,
    "drug_report": drug_report,
    "integrated_shape_report": integrated_shape_report,
    "post_merge_missing_report": post_merge_missing_report,
    "baseline_structure_report": baseline_structure_report,
    "baseline_feature_check": baseline_feature_check,
}

validation_report = make_json_serializable(validation_report)

validation_report_path = INTERIM_DATA_DIR / "data_validation_report.json"

with open(validation_report_path, "w") as f:
    json.dump(validation_report, f, indent=4)

print(f"Data validation report saved to: {validation_report_path}")

Data validation report saved to: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\interim\data_validation_report.json


### 8.3 Check Saved Validation Report

Finally, we load the saved validation report again to confirm that it was written correctly.

In [40]:
with open(validation_report_path, "r") as f:
    loaded_validation_report = json.load(f)

loaded_validation_report.keys()

dict_keys(['file_overview', 'feature_group_summary', 'column_consistency_report', 'duplicate_id_report', 'missing_value_report', 'id_matching_report', 'row_alignment_report', 'missing_extra_id_report', 'target_validity_report', 'metadata_value_report', 'metadata_distribution', 'metadata_consistency_report', 'target_submission_report', 'control_report', 'drug_report', 'integrated_shape_report', 'post_merge_missing_report', 'baseline_structure_report', 'baseline_feature_check'])

## 9. Save Clean Interim Datasets

Now we save the clean and validated datasets to the `data/interim/` folder.

These files will be used in the next notebooks, especially EDA and feature engineering.

We save:

- clean train features,
- clean test features,
- scored target labels,
- nonscored auxiliary target labels,
- drug information,
- merged scored training table,
- merged drug table.

The raw files inside `data/raw/` will remain unchanged.

In [41]:
# Save clean feature tables
train_features.to_parquet(
    INTERIM_DATA_DIR / "train_features_clean.parquet",
    index=False
)

test_features.to_parquet(
    INTERIM_DATA_DIR / "test_features_clean.parquet",
    index=False
)

# Save target tables
train_targets_scored.to_parquet(
    INTERIM_DATA_DIR / "y_scored.parquet",
    index=False
)

train_targets_nonscored.to_parquet(
    INTERIM_DATA_DIR / "y_nonscored.parquet",
    index=False
)

# Save drug table
train_drug.to_parquet(
    INTERIM_DATA_DIR / "train_drug_clean.parquet",
    index=False
)

# Save integrated reference tables
train_main_scored.to_parquet(
    INTERIM_DATA_DIR / "train_main_scored.parquet",
    index=False
)

train_with_drug.to_parquet(
    INTERIM_DATA_DIR / "train_with_drug.parquet",
    index=False
)

print("Clean interim datasets saved successfully.")

Clean interim datasets saved successfully.


### 9.1 Check Saved Interim Files

After saving the clean interim files, we check whether all expected files were created successfully.

This helps us confirm that the next notebook can load these files without problems.

In [42]:
expected_interim_files = [
    "train_features_clean.parquet",
    "test_features_clean.parquet",
    "y_scored.parquet",
    "y_nonscored.parquet",
    "train_drug_clean.parquet",
    "train_main_scored.parquet",
    "train_with_drug.parquet",
    "feature_groups.json",
    "data_validation_report.json",
]

saved_file_check = []

for file_name in expected_interim_files:
    file_path = INTERIM_DATA_DIR / file_name
    
    saved_file_check.append({
        "file_name": file_name,
        "exists": file_path.exists(),
        "file_size_mb": round(file_path.stat().st_size / (1024 * 1024), 3) if file_path.exists() else None,
    })

saved_file_check = pd.DataFrame(saved_file_check)
saved_file_check

,file_name,exists,file_size_mb
0,train_features_clean.parquet,True,96.048
1,test_features_clean.parquet,True,21.802
2,y_scored.parquet,True,0.461
3,y_nonscored.parquet,True,0.577
4,train_drug_clean.parquet,True,0.318
5,train_main_scored.parquet,True,96.258
6,train_with_drug.parquet,True,96.115
7,feature_groups.json,True,0.036
8,data_validation_report.json,True,0.011


### 9.2 Quick Reload Test

To make sure the saved parquet files are readable, we reload a few important files and check their shapes.

This confirms that the saved files are not corrupted and can be used in the next notebooks.

In [43]:
reloaded_train_features = pd.read_parquet(
    INTERIM_DATA_DIR / "train_features_clean.parquet"
)

reloaded_test_features = pd.read_parquet(
    INTERIM_DATA_DIR / "test_features_clean.parquet"
)

reloaded_y_scored = pd.read_parquet(
    INTERIM_DATA_DIR / "y_scored.parquet"
)

reload_check = {
    "reloaded_train_features_shape": reloaded_train_features.shape,
    "reloaded_test_features_shape": reloaded_test_features.shape,
    "reloaded_y_scored_shape": reloaded_y_scored.shape,
    "train_features_shape_match": reloaded_train_features.shape == train_features.shape,
    "test_features_shape_match": reloaded_test_features.shape == test_features.shape,
    "y_scored_shape_match": reloaded_y_scored.shape == train_targets_scored.shape,
}

reload_check

{'reloaded_train_features_shape': (23814, 876),
 'reloaded_test_features_shape': (3982, 876),
 'reloaded_y_scored_shape': (23814, 207),
 'train_features_shape_match': True,
 'test_features_shape_match': True,
 'y_scored_shape_match': True}

## 10. Final Notebook Summary

In this notebook, we completed the data integration and dataset understanding stage for the MoA Prediction project.

### What we completed

We loaded all raw dataset files:

- `train_features.csv`
- `test_features.csv`
- `train_targets_scored.csv`
- `train_targets_nonscored.csv`
- `train_drug.csv`
- `sample_submission.csv`

We checked the basic dataset structure and confirmed that:

- `train_features.csv` has 23,814 rows and 876 columns.
- `test_features.csv` has 3,982 rows and 876 columns.
- `train_targets_scored.csv` has 206 scored target labels plus `sig_id`.
- `train_targets_nonscored.csv` has 402 nonscored target labels plus `sig_id`.
- `sample_submission.csv` contains the same 206 target columns required for final prediction.

### Feature groups identified

The input feature groups are:

- ID column: `sig_id`
- metadata features: `cp_type`, `cp_time`, `cp_dose`
- gene expression features: 772 columns
- cell viability features: 100 columns

The target groups are:

- scored targets: 206 labels
- nonscored targets: 402 labels

### Data quality checks completed

We checked:

- duplicate `sig_id` values,
- missing values,
- train/test column consistency,
- ID matching across files,
- row alignment across files,
- target binary validity,
- metadata category consistency,
- target and submission column consistency,
- control sample target behavior,
- drug ID information,
- repeated drug frequency,
- post-merge missing values.

No critical data quality problems were found.

### Integration completed

We safely created these integrated reference tables:

- `train_main_scored`: training features + scored targets
- `train_with_drug`: training features + drug ID
- `train_aux_nonscored`: sample IDs + nonscored targets

All merged tables kept the correct row count of 23,814 rows.

### Baseline modeling structure

For the baseline model, we prepared the following clean structure:

- `X_train_base`: training features without `sig_id`
- `X_test_base`: test features without `sig_id`
- `y_scored_base`: 206 scored target labels
- `y_nonscored_aux`: 402 nonscored target labels for possible advanced auxiliary learning

The baseline model should not use:

- `sig_id` as a feature,
- `drug_id` as a feature,
- nonscored targets as input features.

### Important modeling decision

For the baseline stage:

```text
Input  = train_features without sig_id
Target = train_targets_scored without sig_id